In [ ]:

%%bash

#
rm -Rf /apps/sandbox/datasets/employee-data/books.xml

#
cat <<EOF > /apps/sandbox/datasets/employee-data/books.xml
<books>
    <book id="bk103">
      <author>Corets, Eva</author>
      <description>A Story Tell</description>
      <genre>Comeady</genre>
      <price>254.00</price>
      <publish_date>2000-07-21</publish_date>
      <title>Maeve Ascendant</title>
    </book>
    <book id="bk104">
      <author>Corets, Eva</author>
      <description>A Love Story</description>
      <genre>Romance</genre>
      <price>136.00</price>
      <publish_date>2020-05-06</publish_date>
      <title>Oberons Legacy</title>
    </book>
</books>
EOF

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName("spark-parquet-data")
    .master("local[*]")
    .config("spark.ui.port", "4040")
    .config("spark.jars.packages", "com.databricks:spark-xml_2.12:0.18.0") 
    .getOrCreate()
)

#
print(spark.sparkContext.uiWebUrl)

#
spark

#
# 
#
def display(df):
    df.show(truncate=False)

:: loading settings :: url = jar:file:/opt/conda/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/brijeshdhaker/.ivy2/cache
The jars for the packages stored in: /home/brijeshdhaker/.ivy2/jars
com.databricks#spark-xml_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-3013ff4c-7334-468e-a896-9acd581a9881;1.0
	confs: [default]
	found com.databricks#spark-xml_2.12;0.18.0 in central
	found commons-io#commons-io;2.11.0 in local-m2-cache
	found org.glassfish.jaxb#txw2;3.0.2 in central
	found org.apache.ws.xmlschema#xmlschema-core;2.3.0 in central
	found org.scala-lang.modules#scala-collection-compat_2.12;2.9.0 in central
downloading https://repo1.maven.org/maven2/com/databricks/spark-xml_2.12/0.18.0/spark-xml_2.12-0.18.0.jar ...
	[SUCCESSFUL ] com.databricks#spark-xml_2.12;0.18.0!spark-xml_2.12.jar (174ms)
downloading file:/home/brijeshdhaker/.m2/repository/commons-io/commons-io/2.11.0/commons-io-2.11.0.jar ...
	[SUCCESSFUL ] commons-io#commons-io;2.11.0!commons-io.jar (21ms)
downloading https://repo1.maven.org/mav

http://vmware-kubuntu.sandbox.net:4040


In [7]:
#
xmlPath = "file:///apps/sandbox/datasets/employee-data/books.xml"

#
df = spark.read.option("rowTag", "books").format("xml").load(xmlPath)
df.printSchema()
df.show(truncate=False)


root
 |-- book: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- _id: string (nullable = true)
 |    |    |-- author: string (nullable = true)
 |    |    |-- description: string (nullable = true)
 |    |    |-- genre: string (nullable = true)
 |    |    |-- price: double (nullable = true)
 |    |    |-- publish_date: date (nullable = true)
 |    |    |-- title: string (nullable = true)

+-----------------------------------------------------------------------------------------------------------------------------------------------------------------+
|book                                                                                                                                                             |
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------+
|[{bk103, Corets, Eva, A Story Tell, Comeady, 254.0, 2000-07-21, Maeve Ascendant

In [8]:
df = spark.read.option("rowTag", "book").format("xml").load(xmlPath)
# Infers three top-level fields and parses `book` in separate rows:
df.show()

+-----+-----------+------------+-------+-----+------------+---------------+
|  _id|     author| description|  genre|price|publish_date|          title|
+-----+-----------+------------+-------+-----+------------+---------------+
|bk103|Corets, Eva|A Story Tell|Comeady|254.0|  2000-07-21|Maeve Ascendant|
|bk104|Corets, Eva|A Love Story|Romance|136.0|  2020-05-06| Oberons Legacy|
+-----+-----------+------------+-------+-----+------------+---------------+



In [9]:
selected_data = df.select("author", "_id")
selected_data.show()

+-----------+-----+
|     author|  _id|
+-----------+-----+
|Corets, Eva|bk103|
|Corets, Eva|bk104|
+-----------+-----+



In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

custom_schema = StructType([
    StructField("_id", StringType(), True),
    StructField("author", StringType(), True),
    StructField("description", StringType(), True),
    StructField("genre", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("publish_date", StringType(), True),
    StructField("title", StringType(), True)
])
df = spark.read.options(rowTag='book').xml('books.xml', schema = custom_schema)

selected_data = df.select("author", "_id")
selected_data.write.options(rowTag='book', rootTag='books').xml('file:///apps/sandbox/datasets/employee-data/newbooks.xml')